# SpectraShift Week 5: freeze downstream contracts
Use CPU with Internet enabled. Attach `spectrashift-source-v4`, `spectrashift-week2-frozen`, and `spectrashift-week4-complete`.


In [ ]:
from pathlib import Path
import json, os, shutil, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week5.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 5 source bundle, found {bundles}'
    source_work = Path('/kaggle/working/week5-source')
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 5 source tree found'
projects.sort(key=lambda path: (0 if 'spectrashift-source' in str(path) else 1, len(str(path))))
PROJECT = projects[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

WORK = Path('/kaggle/working/spectrashift-week5-contracts')
WORK.mkdir(parents=True, exist_ok=True)
manifests = sorted(INPUT.rglob('partitions.parquet'))
normalizations = sorted(INPUT.rglob('normalization.json'))
freeze_summaries = sorted(INPUT.rglob('freeze_summary.json'))
week4_summaries = sorted(INPUT.rglob('week4_run_summary.json'))
assert len(manifests) == len(normalizations) == len(freeze_summaries) == len(week4_summaries) == 1
STAGED = next(path.parent for path in INPUT.rglob('staging_summary.json'))
config = yaml.safe_load((PROJECT / 'configs/downstream/week5.yaml').read_text())
config['data']['manifest_path'] = str(manifests[0])
config['data']['staged_root'] = str(STAGED)
config['data']['normalization_path'] = str(normalizations[0])
config['data']['freeze_summary_path'] = str(freeze_summaries[0])
config['contracts']['output_dir'] = str(WORK)
config['contracts']['week4_summary_path'] = str(week4_summaries[0])
config['contracts']['subset_manifest_path'] = str(WORK / 'downstream_subsets.parquet')
config['contracts']['downstream_contract_path'] = str(WORK / 'downstream_contract.json')
config['contracts']['imagenet_weights_path'] = str(WORK / 'resnet18-f37072fd.pth')
RUNTIME_CONFIG = WORK / 'week5.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print({'project': str(PROJECT), 'work': str(WORK)})


In [ ]:
from spectrashift.train.week5 import prepare_week5_contracts

summary = prepare_week5_contracts(RUNTIME_CONFIG)
print(json.dumps(summary, indent=2))
assert summary['week5_contracts_complete']
assert len(summary['supported_class_indices']) == 16
assert summary['imagenet_weights_sha256'].startswith('f37072fd')
assert summary['evaluation_labels_loaded'] is False
